# Reading assignment A1·R2 — `B4` Ch. 2, *Solving the inference bottleneck with the key-value cache*

**Model Engineering · Course A: Rearchitecting LLMs · module A1**

Chapter 2 of *Build a DeepSeek Model* builds the KV cache from the autoregressive loop
up: it shows the redundancy, proves it, caches it away, prices the cache, and then
introduces MQA and GQA as the first-generation answers to the price. Everything it
demonstrates on GPT-2 you can demonstrate on the reference model — and one thing it only
*times*, you will **count**.

### Learning objectives
1. Write the uncached loop (listing 2.1) and *prove* its redundancy on real activations:
   the first $T$ rows of $K$ at step $T{+}1$ equal all of $K$ at step $T$ (§2.3.2), and
   only the last logits row is ever new (§2.2.3).
2. Build a KV cache for the reference model from scratch, verify it is token-identical to
   the uncached loop, and measure the saving as a FLOP count rather than a stopwatch.
3. Implement the cache-size formula (§2.5.1) and reproduce the book's GPT-2, GPT-3 and
   DeepSeek-V3 figures, then the MQA and GQA reductions (§2.6.2, §2.7.2).
4. Implement GQA as the "tunable knob" (listing 2.4) and show it *is* MHA at one end.

### Sources for this assignment

| Section | Source |
|---|---|
| Autoregressive loop, only-the-last-row insight | `B4` §2.1–2.2 (listing 2.1) |
| Redundancy proof, quadratic vs linear | `B4` §2.3 |
| What to cache and the cached loop | `B4` §2.4 (figures 2.17–2.20) |
| Cache-size formula and the worked numbers | `B4` §2.5 (figure 2.21) |
| MQA and GQA from scratch | `B4` §2.6–2.7 (listings 2.3, 2.4) |
| Reference model | `common/toymodel.py` |

Where the book times (`use_cache=True/False`), this assignment counts FLOPs: on a CPU at
this model size a stopwatch measures Python, not the algorithm — Core M2's lesson.

### Weight — Course A, reading assignment A1·R2

| Component | Marks |
|---|---|
| Implementation (loop, cache, formula, GQA) | 45% |
| Measurement (identity to uncached; FLOP ratio vs prediction) | 35% |
| Interpretation (reflection questions) | 20% |

In [ ]:
import sys, os
# make `common` importable regardless of where Jupyter is launched
_HERE = os.path.abspath("")
for _up in (".", "..", "../.."):
    _cand = os.path.abspath(os.path.join(_HERE, _up))
    if os.path.isdir(os.path.join(_cand, "common")):
        sys.path.insert(0, _cand); break
from common.envcheck import banner, device, has_gpu, tiny_config, GPUS
from common.grading_utils import check, check_close, check_within, scoreboard, GradeError
print(banner())

from common.envcheck import ModelConfig, REFERENCE_CONFIGS
from common.toymodel import load_toy, toy_batch, toy_config, TinyLM, Attention, SEQ
from torch.utils.flop_counter import FlopCounterMode
import copy, math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
PTS = {}

model, cfg = load_toy()
N_PARAMS = sum(p.numel() for p in model.parameters())
idx_all = toy_batch(1, seed=11)[0]          # one 64-token sequence; the toy's context limit
print(f"toy: {cfg.n_layers} layers, {cfg.n_heads} heads, {cfg.n_kv_heads} KV heads, "
      f"d_head {cfg.d_head}, context {SEQ}")

### Shapes and symbols

Used consistently in every docstring and shape annotation in this notebook.

| symbol | meaning | where it comes from |
|---|---|---|
| `B` | batch: sequences processed together | your call |
| `T` | sequence length, `T <= SEQ` | your input |
| `d` | residual-stream width | `cfg.d_model` |
| `d_ff` | MLP expansion width — the **width** axis | `cfg.d_ff` |
| `n_layers` | transformer blocks — the **depth** axis | `cfg.n_layers` |
| `h` | query heads per block | `cfg.n_heads` → `Attention.h` |
| `hkv` | KV heads, i.e. KV **groups** | `cfg.n_kv_heads` → `Attention.hkv` |
| `dh` | per-head width, `d // h` | `cfg.d_head` → `Attention.dh` |
| `rep` | query heads per KV group, `h // hkv` | derived |
| `vocab` | vocabulary size | `cfg.vocab` |

`hkv` is the one to keep straight. One KV group's keys and values are shared by `rep`
consecutive query heads, so query head `q` reads group `q // rep`. `hkv == h` is **MHA**
(every head owns its K/V), `hkv == 1` is **MQA** (all heads share one pair), anything
between is **GQA**. It is also the single number that sets the KV-cache budget.

**Tensor layouts**

| tensor | shape |
|---|---|
| token ids `idx`, targets `tgt`, boolean masks | `(B, T)` |
| hidden state `x` | `(B, T, d)` |
| logits | `(B, T, vocab)` |
| `q.weight` / `o.weight` | `(h*dh, d)` / `(d, h*dh)` |
| `k.weight`, `v.weight` | `(hkv*dh, d)` |
| `gate.weight`, `up.weight` / `down.weight` | `(d_ff, d)` / `(d, d_ff)` |
| q inside attention / k, v inside attention | `(B, h, T, dh)` / `(B, hkv, T, dh)` |

Query head `q` owns rows `q*dh : (q+1)*dh` of `q.weight` and the matching **columns** of
`o.weight`. MLP neuron `i` owns row `i` of `gate` and `up` but **column** `i` of `down`.
Those two row/column asymmetries are where most pruning bugs come from.

## Task 1 — The uncached loop, and the redundancy made visible

Listing 2.1 feeds the *whole* sequence back through the model at every step and keeps
only the last logits row. Write it. Then make the chapter's two observations into
assertions:

- **§2.2.3** — the logits for the old positions do not change when a token is appended
  (causal attention: nothing later can reach them).
- **§2.3.2** — the first $T$ rows of the layer-0 Key matrix at length $T{+}1$ are
  *identical* to the whole Key matrix at length $T$. Capture $K$ with a forward hook on
  the layer's `k` projection; do not recompute it by hand.

In [ ]:
@torch.no_grad()
def generate_uncached(model, idx, n_new):
    # TODO(a1r2.1): listing 2.1 -- the whole sequence through the model at every step.
    #
    #   Args:  model -- (B, T) ids -> (B, T, vocab) logits.
    #          idx   -- (B, T0) int64 prompt.
    #          n_new -- tokens to append, greedily (argmax, so deterministic).
    #   Returns: (B, T0 + n_new) int64. Each step re-runs the model over every position
    #            and keeps only logits[:, -1, :] -- that discarded work is what this
    #            assignment measures. Needs T0 + n_new <= SEQ.
    raise NotImplementedError("TODO(a1r2.1): implement generate_uncached")


@torch.no_grad()
def k_rows(model, idx, layer=0):
    # TODO(a1r2.1): capture one layer's K projection where it is computed.
    #
    #   Args:  model -- the model to run.
    #          idx   -- (B, T) int64 ids.
    #          layer -- block index whose attn.k output to capture.
    #   Returns: (B, T, hkv*dh) -- the RAW projection, before any reshape to
    #            (B, hkv, T, dh). One row per token: exactly what a cache stores.
    #   Note:  register a forward hook on model.blocks[layer].attn.k and remove it again
    #          before returning, so the model is left as you found it.
    raise NotImplementedError("TODO(a1r2.1): implement k_rows")

In [ ]:
T0, N_NEW = 16, 24
prompt = idx_all[:, :T0]
gen_unc = generate_uncached(model, prompt, N_NEW)
print("uncached generation:", gen_unc[0].tolist())

with torch.no_grad():
    l4, l5 = model(idx_all[:, :4]), model(idx_all[:, :5])
    k4, k5 = k_rows(model, idx_all[:, :4]), k_rows(model, idx_all[:, :5])
same_logits = torch.allclose(l5[:, :4], l4, atol=1e-5)
same_k = torch.allclose(k5[:, :4], k4, atol=1e-6)
print(f"\nlogits rows 0..3 unchanged after appending a token : {same_logits}")
print(f"K rows 0..3 unchanged after appending a token      : {same_k}")

PTS["a1r2.1 loop"] = check(
    gen_unc.shape == (1, T0 + N_NEW) and torch.equal(gen_unc[:, :T0], prompt),
    "generate_uncached extends the prompt by n_new greedy tokens", 2)
PTS["a1r2.1 only the last row is new"] = check(same_logits, "old logits rows do not move (§2.2.3)", 2)
PTS["a1r2.1 K is recomputed identically"] = check(same_k, "old K rows are recomputed bit-for-bit (§2.3.2)", 3)

## Task 2 — A KV cache for the reference model, and the saving counted

§2.4.2 gives the cached loop in eight steps. Implement it for `TinyLM`:

- `attn_step(attn, x, cache)` — attention over the **new** tokens only, against the
  cached keys and values plus the new ones. Write the attention with explicit matmuls
  rather than `F.scaled_dot_product_attention`: on CPU the fused kernel is invisible to
  `FlopCounterMode`, and Task 2 needs to *count*.
- `model_step(model, idx_new, pos_offset, caches)` — embeddings need the **absolute
  position** (the toy uses a learned table; a RoPE model needs the same offset, see `B5`
  App. C), then each block with its own cache.
- `generate_cached(model, idx, n_new)` — prefill once, then one token per step.

The cache is per layer (and per head inside the tensors), exactly as §2.4.2 says.

**Predict before you measure.** All linear layers do work proportional to the number of
tokens they process. Uncached, step $i$ processes $T_0 + i$ tokens; cached, the prefill
processes $T_0$ and every later step processes one. The prediction cell writes that ratio
down; the measurement cell then counts.

In [ ]:
@torch.no_grad()
def attn_step(attn, x, cache):
    # TODO(a1r2.2): attention for T new tokens against cached + new K/V (§2.4.2 steps 2-6).
    #
    #   Args:  attn  -- an Attention module; read its weights and .h, .hkv, .dh.
    #          x     -- (B, T, d), ALREADY normalised, for the NEW positions only.
    #                   T is the prompt length on prefill and 1 during decode.
    #          cache -- None on the first call, else (k, v) each (B, hkv, S_past, dh).
    #   Returns: (out, new_cache) where
    #            out       -- (B, T, d) for the new positions.
    #            new_cache -- (k, v) each (B, hkv, S_past + T, dh). Build it BEFORE any
    #                         GQA repeat_interleave: the cache is per KV HEAD, which is
    #                         why hkv and not h sets the budget.
    #   Shapes:  q is (B, h, T, dh); k, v are (B, hkv, S, dh) with S = S_past + T.
    #            scores = q @ k^T / sqrt(dh) is (B, h, T, S).
    #   Note:  mask key j for query i when j > past + i, where past = S - T. Row i is
    #          absolute position past + i, so a new token must still see the whole cached
    #          prefix -- a plain upper-triangular mask silently generates plausible but
    #          different text.
    raise NotImplementedError("TODO(a1r2.2): implement attn_step")


@torch.no_grad()
def model_step(model, idx_new, pos_offset, caches):
    # TODO(a1r2.2): push the new tokens through every block, updating the caches.
    #
    #   Args:  model      -- the model to run.
    #          idx_new    -- (B, T) int64 ids for the NEW positions only.
    #          pos_offset -- absolute index of the first new token, so the positional
    #                        embedding stays correct across steps. During decode this is
    #                        (tokens so far) - 1.
    #          caches     -- list of n_layers entries, each None or (k, v). MUTATED IN
    #                        PLACE. Pass [None] * n_layers to start fresh, and never
    #                        reuse one list across two independent generations.
    #   Returns: (B, T, vocab) logits for the new positions only. During decode T is 1,
    #            so logits[:, -1, :] is the only row there is.
    raise NotImplementedError("TODO(a1r2.2): implement model_step")


@torch.no_grad()
def generate_cached(model, idx, n_new):
    # TODO(a1r2.2): prefill once, then one token per step (§2.4.2).
    #
    #   Args:  model -- the model to run.
    #          idx   -- (B, T0) int64 prompt.
    #          n_new -- tokens to append, greedily.
    #   Returns: (out, caches) where
    #            out    -- (B, T0 + n_new) int64. Must be TOKEN-IDENTICAL to
    #                      generate_uncached; that equality is the correctness test.
    #            caches -- per-layer (k, v), each (B, hkv, T0 + n_new - 1, dh).
    #   Note:  the cache ends up with T0 + n_new - 1 rows, not T0 + n_new. The final
    #          sampled token is appended to the output but never fed back, so it is never
    #          projected to K/V.
    raise NotImplementedError("TODO(a1r2.2): implement generate_cached")

In [ ]:
# PREDICTION -- written before the measurement below. Reasoning:
#   every Linear does work proportional to tokens processed, so the uncached/cached FLOP
#   ratio should be close to (sum over steps of tokens re-processed) / (tokens processed
#   once). The attention-score matmuls are quadratic in length, so the measured ratio
#   should come out slightly ABOVE this linear-only prediction.
tokens_uncached = sum(T0 + i for i in range(N_NEW))
tokens_cached = T0 + N_NEW - 1
predicted_ratio = tokens_uncached / tokens_cached
print(f"tokens projected, uncached : {tokens_uncached}")
print(f"tokens projected, cached   : {tokens_cached}")
print(f"PREDICTION: uncached/cached FLOP ratio ~ {predicted_ratio:.2f} (a little more, from the scores)")

In [ ]:
def count_flops(fn):
    """Total FLOPs executed by `fn()`, via torch's `FlopCounterMode`.

    Args:
        fn: zero-argument callable, invoked exactly once inside the counter.

    Returns:
        int FLOP count. Matmuls only -- elementwise work and softmax are not counted, which
        is what makes it the right instrument for comparing matmul-bound paths.
    """
    fc = FlopCounterMode(display=False)
    with fc:
        fn()
    return fc.get_total_flops()

# identity to the uncached loop, on three prompts
ok = True
for seed in (11, 12, 13):
    p = toy_batch(1, seed=seed)[0][:, :T0]
    ok &= torch.equal(generate_cached(model, p, N_NEW)[0], generate_uncached(model, p, N_NEW))
gen_c, caches = generate_cached(model, prompt, N_NEW)
cache_len = caches[0][0].shape[2]

# measured work: both paths go through the same explicit-matmul code, so the counter
# sees the same operations; the uncached path simply re-runs the growing prefix.
flops_cached = count_flops(lambda: generate_cached(model, prompt, N_NEW))
flops_uncached = sum(count_flops(lambda: model_step(model, gen_c[:, :T0 + i], 0, [None] * cfg.n_layers))
                     for i in range(N_NEW))
measured_ratio = flops_uncached / flops_cached
print(f"cached == uncached on 3 prompts : {ok}")
print(f"cache length after {N_NEW} tokens  : {cache_len} (= T0 + n_new - 1)")
print(f"FLOPs uncached : {flops_uncached:,}")
print(f"FLOPs cached   : {flops_cached:,}")
print(f"measured ratio : {measured_ratio:.2f}   predicted {predicted_ratio:.2f}")

PTS["a1r2.2 cached is token-identical"] = check(ok, "KV-cached generation reproduces the uncached tokens", 5)
PTS["a1r2.2 cache holds T0+n_new-1"] = check(cache_len == tokens_cached, "one K/V row per token, per layer", 2)
PTS["a1r2.2 saving matches prediction"] = check_within(
    measured_ratio, predicted_ratio, 0.15, "counted FLOP ratio is within 15% of the linear-work prediction", 4)
PTS["a1r2.2 scores push it above"] = check(
    measured_ratio > predicted_ratio, "the quadratic score matmuls make the uncached path worse still", 1)

## Task 3 — The dark side: the cache-size formula

§2.5.1 (figure 2.21):

$$\text{bytes} = l \cdot b \cdot n_{kv} \cdot h \cdot s \cdot 2 \cdot \text{bytes/param}$$

The chapter's worked numbers are all at $s = 1024$, $b = 1$, 16-bit, binary units for the
GPT-2/GPT-3 pair and decimal for the two large examples: GPT-2 (12 layers, 12 heads of 64)
**36 MB**; GPT-3 (96 layers, 96 heads of 128) **4.5 GB**; a 30B model (48 layers,
$n{\cdot}h = 7168$, $b = 128$) **180 GB**; DeepSeek-V3 (61 layers, 128 heads of 128,
$s = 100{,}000$) **≈ 400 GB**. Then §2.6.2: MQA divides GPT-3's cache by 96 to **48 MB**;
§2.7.2: Llama-3-8B's 32 → 8 KV heads divides it by **4**.

Reproduce all of them from one function, then price the toy's own cache.

In [ ]:
def kv_cache_bytes(layers, batch, kv_heads, head_dim, seq, bytes_per=2):
    # TODO(a1r2.3): figure 2.21 -- KV-cache bytes.
    #
    #   Args:  layers     -- transformer blocks, each with its own cache.
    #          batch      -- concurrent sequences.
    #          kv_heads   -- KV heads per block, i.e. hkv. The QUERY-head count does not
    #                        appear; that absence is the whole saving of GQA/MQA.
    #          head_dim   -- per-head width, dh.
    #          seq        -- tokens cached per sequence.
    #          bytes_per  -- bytes per stored value: 2 for fp16/bf16, 4 for fp32.
    #   Returns: int bytes. Remember the factor 2 for K and V.
    raise NotImplementedError("TODO(a1r2.3): implement kv_cache_bytes")


def kv_bytes_per_token(cfg, bytes_per=2):
    # TODO(a1r2.3): the marginal KV cost of one extra token, for one sequence.
    #
    #   Args:  cfg       -- a ModelConfig; read n_layers, n_kv_heads and d_head.
    #          bytes_per -- bytes per stored value.
    #   Returns: int bytes per token. Multiply by context length for one request, or by
    #            batch * context to size a server.
    raise NotImplementedError("TODO(a1r2.3): implement kv_bytes_per_token")

In [ ]:
MiB, GiB = 2 ** 20, 2 ** 30
gpt2 = kv_cache_bytes(12, 1, 12, 64, 1024)
gpt3 = kv_cache_bytes(96, 1, 96, 128, 1024)
big30 = kv_cache_bytes(48, 128, 56, 128, 1024)            # n*h = 7168 = 56 x 128
dsv3 = kv_cache_bytes(61, 1, 128, 128, 100_000)
gpt3_mqa = kv_cache_bytes(96, 1, 1, 128, 1024)
l3 = REFERENCE_CONFIGS["llama3-8b"]
l3_mha = ModelConfig("llama3-8b-mha", l3.n_layers, l3.d_model, l3.n_heads, l3.n_heads, l3.d_ff, l3.vocab)
gqa_factor = kv_bytes_per_token(l3_mha) / kv_bytes_per_token(l3)

print(f"GPT-2        : {gpt2 / MiB:7.1f} MiB     (book: 36 MB)")
print(f"GPT-3        : {gpt3 / GiB:7.2f} GiB     (book: 4.5 GB)")
print(f"30B, b=128   : {big30 / 1e9:7.1f} GB      (book: 180 GB)")
print(f"DeepSeek-V3  : {dsv3 / 1e9:7.1f} GB      (book: ~400 GB, one sequence at 100k)")
print(f"GPT-3 + MQA  : {gpt3_mqa / MiB:7.1f} MiB     (book: 48 MB)")
print(f"Llama-3-8B   : MHA / GQA cache = {gqa_factor:.0f}x   (book: 4)")

toy_kv = kv_bytes_per_token(cfg, bytes_per=4)
toy_w = N_PARAMS * 4
print(f"\ntoy at fp32  : {toy_kv} B per token; weights {toy_w:,} B; "
      f"cache overtakes weights at s = {toy_w / toy_kv:,.0f} tokens (context is {SEQ})")

PTS["a1r2.3 GPT-2 and GPT-3"] = check(
    gpt2 == 36 * MiB and abs(gpt3 - 4.5 * GiB) < 1, "36 MiB and 4.5 GiB, exactly", 3)
PTS["a1r2.3 the two large examples"] = check(
    abs(big30 / 180e9 - 1) < 0.01 and abs(dsv3 / 400e9 - 1) < 0.01,
    "180 GB and ~400 GB (decimal), within 1%", 2)
PTS["a1r2.3 MQA and GQA factors"] = check(
    gpt3_mqa == 48 * MiB and gqa_factor == 4.0, "MQA divides by heads; Llama-3-8B GQA divides by 4", 3)

## Task 4 — GQA as the tunable knob

Listing 2.4 is MQA (listing 2.3) with `num_groups` K/V projections instead of one, and
`repeat_interleave` to hand each group to its query heads. Implement it **bias-free** so
that it can load the toy's trained attention weights, then show the two ends of the knob:

- `num_groups = num_heads` is MHA — loaded with the toy's weights it must reproduce the
  toy's attention output;
- `num_groups = 1` is MQA — its K/V parameters are `1/num_heads` of MHA's.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Listing 2.4 without biases or dropout."""
    def __init__(self, d_model, num_heads, num_groups):
        super().__init__()
        # TODO(a1r2.4): build listing 2.4, bias-free.
        #
        #   The book's naming: num_groups here is the same quantity the rest of the
        #   course calls hkv.
        #
        #   Args:  d_model    -- residual width d.
        #          num_heads  -- query heads, h.
        #          num_groups -- KV groups, hkv. Assert that num_groups divides
        #                        num_heads AND that num_heads divides d_model.
        #   Attributes to create:
        #     self.d_head -- d_model // num_heads, i.e. dh.
        #     self.W_q, self.W_o -- (d_model, d_model).
        #     self.W_k, self.W_v -- (num_groups * d_head, d_model). The knob: these are
        #                           the only two projections GQA shrinks.
        raise NotImplementedError("TODO(a1r2.4): implement GroupedQueryAttention.__init__")

    def forward(self, x):
        # TODO(a1r2.4): (B, T, d_model) -> (B, T, d_model), causally masked.
        #
        #   Shapes:  q is (B, num_heads, T, d_head); k and v start (B, num_groups, T,
        #            d_head) and are repeat_interleave(rep, dim=1)-ed to num_heads, where
        #            rep = num_heads // num_groups. scores is (B, num_heads, T, T).
        #   Note:  dim=1 is the head axis. repeat_interleave there gives group g to heads
        #          g*rep .. (g+1)*rep-1; repeating along the wrong axis, or with .repeat
        #          instead of .repeat_interleave, pairs heads with the wrong groups and
        #          still returns the right SHAPE.
        raise NotImplementedError("TODO(a1r2.4): implement GroupedQueryAttention.forward")


def load_from_toy(gqa, attn):
    # TODO(a1r2.4): carry the toy's attention weights into your GQA module, in place.
    #
    #   Args:  gqa  -- destination GroupedQueryAttention.
    #          attn -- source Attention, which must be MHA.
    #   Returns: gqa, mutated -- so the two can be checked for identical output.
    #   Raises:  AssertionError unless gqa.num_groups == gqa.num_heads == attn.h. Only
    #            MHA weights transfer directly: a real GQA module has fewer K/V rows than
    #            an MHA source provides, so no shape-correct copy exists.
    #   Note:  copy under torch.no_grad().
    raise NotImplementedError("TODO(a1r2.4): implement load_from_toy")

In [ ]:
d, h = cfg.d_model, cfg.n_heads
xin = torch.randn(2, 32, d)
attn0 = model.blocks[0].attn
mha = load_from_toy(GroupedQueryAttention(d, h, h), attn0)
with torch.no_grad():
    err = (mha(xin) - attn0(xin)).abs().max().item()
kv_params = lambda m: m.W_k.weight.numel() + m.W_v.weight.numel()
kv = {g: kv_params(GroupedQueryAttention(d, h, g)) for g in (4, 2, 1)}
print(f"groups = heads : output error vs the toy's attention {err:.2e}")
print("K/V parameters :", {f"g={g}": v for g, v in kv.items()})

PTS["a1r2.4 the knob at MHA"] = check(err < 1e-5, "num_groups == num_heads reproduces MHA exactly", 4)
PTS["a1r2.4 the knob at MQA"] = check(
    kv[4] / kv[1] == h and kv[4] / kv[2] == 2, "K/V parameters scale with the number of groups", 3)

## Grading

In [ ]:
scoreboard(PTS, total_possible=34)

## Reflection questions

1. **Why K and V, not Q.** §2.4.1 derives what to cache by working backwards from the one
   context vector you need. Restate that derivation in two sentences and say what would
   go wrong if you cached Q "for symmetry".
2. **Bandwidth, not capacity.** §2.5 says the cached decode step becomes *memory-bandwidth
   bound*. For GPT-3 at $s = 1024$, $b = 1$, compare the bytes of KV cache read per step
   with the bytes of weights read per step. Which dominates, and at what batch size does
   that flip?
3. **Counting versus timing.** The book reports a 5× wall-clock speedup for 100 tokens.
   Your counted ratio for 24 tokens was different. List the reasons the two numbers
   should *not* agree, and which one you would put in a design document.
4. **`.repeat()` versus `repeat_interleave` versus `expand`.** Listing 2.3 says
   `.repeat()` "doesn't actually copy data". Check the claim (`untyped_storage().data_ptr()`
   or `is_contiguous()` / `stride()`), and say which of the three you would use in a
   production cache and why.